In [1]:
# Importing Modules
import numpy as np
import pandas as pd
from src.utils.smiles2morganfp import MorganFP
from src.gnn_fp_utils.dataloader import loadData
from src.gnn_fp_utils.model import GNNModel
from src.train_test import TrainGNN, TestGNN
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/processed/train/ESOL.csv")
esol_test_data = pd.read_csv("data/processed/test/ESOL.csv")

# Generate ESOL FP
esol_train_fp = MorganFP(esol_train_data["smiles"])
esol_train_fp["smiles"] = esol_train_fp.index
esol_train_fp = esol_train_fp.merge(esol_train_data, on="smiles")
esol_test_fp = MorganFP(esol_test_data["smiles"])
esol_test_fp["smiles"] = esol_test_fp.index
esol_test_fp = esol_test_fp.merge(esol_test_data, on="smiles")

######################## DATA-2 ##################################
# Loading RT data
rt_train_data = pd.read_csv("data/processed/train/RT.csv")
rt_test_data = pd.read_csv("data/processed/test/RT.csv")

# Generate RT FP
rt_train_fp = MorganFP(rt_train_data["smiles"])
rt_train_fp["smiles"] = rt_train_fp.index
rt_train_fp = rt_train_fp.merge(rt_train_data, on="smiles")
rt_test_fp = MorganFP(rt_test_data["smiles"])
rt_test_fp["smiles"] = rt_test_fp.index
rt_test_fp = rt_test_fp.merge(rt_test_data, on="smiles")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/processed/train/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/processed/test/Lipophilicity.csv")

# Generate lipophilicity FP
lipophilicity_train_fp = MorganFP(lipophilicity_train_data["smiles"])
lipophilicity_train_fp["smiles"] = lipophilicity_train_fp.index
lipophilicity_train_fp = lipophilicity_train_fp.merge(lipophilicity_train_data, on="smiles")
lipophilicity_test_fp = MorganFP(lipophilicity_test_data["smiles"])
lipophilicity_test_fp["smiles"] = lipophilicity_test_fp.index
lipophilicity_test_fp = lipophilicity_test_fp.merge(lipophilicity_test_data, on="smiles")

######################## DATA-4 ##################################
# Loading B3DB data
b3db_train_data = pd.read_csv("data/processed/train/B3DB.csv")
b3db_test_data = pd.read_csv("data/processed/test/B3DB.csv")

# Generate B3DB FP
b3db_train_fp = MorganFP(b3db_train_data["smiles"])
b3db_train_fp["smiles"] = b3db_train_fp.index
b3db_train_fp = b3db_train_fp.merge(b3db_train_data, on="smiles")
b3db_test_fp = MorganFP(b3db_test_data["smiles"])
b3db_test_fp["smiles"] = b3db_test_fp.index
b3db_test_fp = b3db_test_fp.merge(b3db_test_data, on="smiles")

In [3]:
# Function to run analysis
def RunGNN(train_data, test_data, dataName, modelName, params):

    # Params
	h_dim, b_size, lr, d_out, wd = params

	# Loading dataset
	train_loader = loadData(train_data, batch_size=b_size)
	test_loader = loadData(test_data, batch_size=b_size)

	# Model
	model = GNNModel(num_features=6, hidden_dim=h_dim, model_type=modelName, dropout=d_out)

	# Model training
	model = TrainGNN(model, train_loader, epochs=100, learning_rate=lr, w_decay=wd)

	# Model testing
	y_test, y_pred = TestGNN(model, test_loader)

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)
    
    # Bootstrap 95% CI
	rng = np.random.default_rng(42)
	mae_samples = []
	rmse_samples = []

	y_test_arr = np.array(y_test)
	y_pred_arr = np.array(y_pred)

	for _ in range(2000):
		idx = rng.integers(0, len(y_test_arr), len(y_test_arr))
		mae_samples.append(mean_absolute_error(y_test_arr[idx], y_pred_arr[idx]))
		rmse_samples.append(root_mean_squared_error(y_test_arr[idx], y_pred_arr[idx]))

	mae_lower, mae_upper = np.percentile(mae_samples, [2.5, 97.5])
	rmse_lower, rmse_upper = np.percentile(rmse_samples, [2.5, 97.5])

	# Return results
	return pd.DataFrame({ "Data": [dataName], "Model": [modelName],
                          "MAE": [mae], "MAE_lower": [mae_lower],
                          "MAE_upper": [mae_upper], "RMSE": [rmse],
                          "RMSE_lower": [rmse_lower], "RMSE_upper": [rmse_upper]
                        })


In [4]:
# Data dict
datasets = {"ESOL":{"train":esol_train_fp, "test":esol_test_fp},
            "Lipophilicity":{"train":lipophilicity_train_fp, "test":lipophilicity_test_fp},
            "RT":{"train":rt_train_fp, "test":rt_test_fp},
            "B3DB":{"train":b3db_train_fp, "test":b3db_test_fp}}

# Models
models = ["GCN", "GAT", "GIN", "GraphSAGE"]

In [5]:
# List to store results
temp_out = []

# Loop for models
for modelName in models:
    # Loop for dataset
    for dataName, data in datasets.items():
        # Extract params
        temp_df = pd.read_csv(f"results/Output_Hyperparameter_Optimization_GNN_FP_{dataName}.csv")
        temp = temp_df[temp_df["Model"]==modelName]
        params = temp.sort_values(by=["RMSE"]).head(1)[["h_dim", "b_size", "lr", "d_out", "w_decay"]].to_dict('records')[0].values()
        # Run Analysis for model and dataset
        temp_out.append(RunGNN(data["train"], data["test"], dataName, modelName, params))

# Final output
GNN_out = pd.concat(temp_out, ignore_index=True)
GNN_out.to_csv("results/Output_GNN_FP_Analysis.csv")
GNN_out

,Data,Model,MAE,MAE_lower,MAE_upper,RMSE,RMSE_lower,RMSE_upper
0,ESOL,GCN,0.816234,0.718087,0.914365,1.073080,0.946999,1.192714
1,Lipophilicity,GCN,0.789238,0.703347,0.876875,1.022236,0.917297,1.122362
2,RT,GCN,72.683542,63.079393,82.783719,102.744425,88.063714,117.234478
3,B3DB,GCN,0.430770,0.379396,0.489069,0.586561,0.512086,0.663511
4,ESOL,GAT,0.770234,0.669844,0.868367,1.041214,0.910987,1.161382
5,Lipophilicity,GAT,0.779947,0.698269,0.866391,0.993080,0.892377,1.089447
6,RT,GAT,71.284948,61.103225,81.392925,101.402229,86.197666,115.945709
7,B3DB,GAT,0.417534,0.363796,0.474526,0.586316,0.512780,0.666636
8,ESOL,GIN,0.792261,0.697855,0.888175,1.054263,0.931955,1.172866
9,Lipophilicity,GIN,0.804235,0.713510,0.902420,1.037683,0.927222,1.143904


In [6]:
# Function to plot barplot showing result
def plot_bar(data, target):
    data = data.copy()
    data["err_lower"] = data[target] - data[f"{target}_lower"]
    data["err_upper"] = data[f"{target}_upper"] - data[target]
    sns.set_theme(style="ticks", context="paper")
    model_order = data["Model"].unique()
    g = sns.catplot(
        data=data, kind="bar", x="Model", y=target, hue="Model",
        col="Data", col_wrap=4, sharey=False, height=4, width=0.8, 
        aspect=0.8, dodge=False, errorbar=None, order=model_order)
    for i, ax in enumerate(g.axes.flat):
        col_name = g.col_names[i]
        subdata = data[data["Data"] == col_name].set_index("Model").reindex(model_order).reset_index()
        if not subdata.dropna(subset=[target]).empty:
            ax.errorbar(
                x=np.arange(len(subdata)), 
                y=subdata[target],
                yerr=[subdata["err_lower"], subdata["err_upper"]], 
                fmt="none", capsize=4, linewidth=1, color="black")
    g.set_axis_labels("Model", f"{target}")
    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=40)
    plt.tight_layout()
    plt.savefig(f"results/GNN_FP_Analysis_{target}_Plot.png", dpi=300)
    plt.close()

In [7]:
# Plotting barplot for RMSE
plot_bar(GNN_out, "RMSE")
# Plotting barplot for MAE
plot_bar(GNN_out, "MAE")